# 01. Ingest Raw Clearinghouse Transaction Logs
**Cymbal Financial Fraud Detection Pipeline**

This notebook ingests unstructured JSON transaction logs from Cloud Storage (`gs://${PROJECT_ID}-fin-clearing-raw/`) into the BigQuery raw layer (`transactions_dataset_evals.raw_transactions`) using Apache Spark.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp

# Initialize or retrieve active Spark Session
spark = SparkSession.builder \
    .appName("Cymbal-Fraud-Ingestion") \
    .getOrCreate()

project_id = os.getenv("PROJECT_ID", "cymbal-fraud-detection")
dataset_name = os.getenv("DATASET_NAME", "transactions_dataset_evals")
raw_bucket = os.getenv("RAW_BUCKET", f"{project_id}-fin-clearing-raw")

# Determine input path (Cloud Storage vs Local fallback)
gcs_path = f"gs://{raw_bucket}/logs.json"
local_path = "data/logs.json"
input_path = gcs_path if os.path.exists("/var/log/dataproc") or "gs://" in gcs_path and not os.path.exists(local_path) else local_path

print(f"Reading transaction logs from: {input_path}")

In [ ]:
# Read the raw JSON transaction logs
raw_df = spark.read.json(input_path)

print(f"Loaded {raw_df.count():,} raw records")
raw_df.printSchema()
raw_df.show(5, truncate=False)

In [ ]:
# Write raw records to BigQuery using BigQuery connector (or local parquet/delta table)
target_table = f"{project_id}.{dataset_name}.raw_transactions"
print(f"Writing records to {target_table} with overwrite mode...")

try:
    raw_df.write \
        .format("bigquery") \
        .option("table", target_table) \
        .option("temporaryGcsBucket", f"{project_id}-airflow-artifacts") \
        .mode("overwrite") \
        .save()
    print("Successfully written to BigQuery.")
except Exception as e:
    print(f"Cloud BigQuery write bypassed (Local Mode): {e}")
    # Save locally for verification
    output_dir = "data/raw_transactions_parquet"
    raw_df.write.mode("overwrite").parquet(output_dir)
    print(f"Saved locally to {output_dir}")

In [ ]:
# Verification and Summary Statistics
print("=== Ingestion Summary ===")
print(f"Total Ingested Transactions: {raw_df.count():,}")
print(f"Unique Payment Methods: {raw_df.select('payment_method').distinct().count()}")
print(f"Labeled Records: {raw_df.filter(col('is_fraud').isNotNull()).count():,}")
print(f"Unlabeled Records (for inference): {raw_df.filter(col('is_fraud').isNull()).count():,}")